In [4]:
import os
import gc
import re
import time
from itertools import zip_longest
from datetime import datetime
from joblib import Parallel, delayed

import numpy as np
import pandas as pd


"""
STVM 시계열 + 정규화(Z-score, 환산점수) 산출
- STVM 계산 함수(2~4장)는 기존 ToyNetwork 재분석 코드 그대로
- 5장 Z-score 환산 함수(z_to_score)는 1. stvm__hwaseong_seoul_V1.0.ipynb 그대로
- 정규화 기준: 시나리오별 (한 시나리오 파일의 전 위치 x 전 시간 STVM으로 평균·STDEV.P, Z최소·Z최대)
- 출력: 시나리오별 CSV
  scenario_id, time_sec, detector_location_id, STVM, Z_score, 환산점수, aggregation_sec, run_type, notes
"""


# =============================================================================
# 1. 입력값 설정  ← 경로·대상은 여기만 수정
# =============================================================================
# 원시 parquet 최상위 폴더 ※ 세트·구간 폴더까지 넣지 말 것
#   (이 아래에 "1. 추정용 시나리오\진입구간" 처럼 세트\구간 폴더가 자동으로 붙음)
RAW_ROOT = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\시나리오(140-추정용,검증용)_261022"

# 시나리오 CSV 최상위 폴더 (이 아래에 "1. 추정용 시나리오", "2. 검증용 시나리오")
SCN_ROOT = r"E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\유고 시나리오"

# 결과 저장: 각 parquet 폴더 안에 이 이름의 폴더를 만들고 그 안에 바로 CSV 저장
#   예) ...\1. 추정용 시나리오\진입구간\STVM 시계열\Scenario_001_10회평균_STVM_진입부.csv
RESULT_FOLDER_NAME = "STVM 시계열"

# 세트: 이름 → 폴더명
SET_DIRS = {"추정용": "1. 추정용 시나리오", "검증용": "2. 검증용 시나리오"}
RUN_SETS = ["추정용"]                    # 예: ["추정용", "검증용"]

# 구간: (원시 하위 폴더, 시나리오 CSV 파일명, 출력 파일명 끝)
#   연결로 영향권은 "_3차로폐쇄" 파일을 사용 (그동안 쓴 연결로-영향권 CSV와 내용 동일 확인)
_CSV = "synthetic_scenarios_all_140_with_set_type({}).csv"
SECTIONS = (
    [("진입구간", _CSV.format("진입구간"), "진입부")]
    + [(rf"본선부\구간{i}", _CSV.format(f"본선부-구간{i}"), f"본선부_구간{i}") for i in range(1, 6)]
    + [(rf"연결로 영향권\영향권{i}",
        f"synthetic_scenarios_all_140_with_set_type(영향권{i}({k}))_3차로폐쇄.csv",
        f"연결로 영향권{i}_{k}")
       for i, k in zip(range(1, 6), ["진출", "진입", "진출", "진입", "진출"])]
    + [("진출구간", _CSV.format("진출구간"), "진출부")]
)
# 원시 하위 폴더 이름으로 고름. None = 12개 구간 전부
#   예: ["진입구간"], [r"본선부\구간1"], [r"연결로 영향권\영향권3"]
RUN_SECTIONS = ["진입구간"]              # None = 12개 구간 전부
RUN_SCENARIOS = [1]                     # None = 전부, 시험할 때는 [1]

# 한 시나리오당 seed 파일 수
num_mer = 10

# 집계시간: 초 단위
SEC_TIME = 10

# 동시에 처리할 원시 파일 수 (1이면 순차 처리, joblib loky 사용 → Jupyter에서도 작동)
N_WORKERS = 10

# 시나리오 CSV 저장 후 그 시나리오의 processed 파일 삭제 여부
DELETE_PROCESSED = True

start_interval = 1800
end_use_interval = 12600
end_interval = 13200

acc_start_time = 3600
Vc = 53.7

weights = {
    "w1": 1,
    "w2": 1,
    "w3": 1,
    "w4": 1,
    "w5": 1,
    "w6": 1,
}

vehicle_types = [100, 300, 630, 640, 650]

# 램프 관련 검지기
before_ramp = [70, 117, 124, 186, 215]
after_ramp = [74, 121, 127, 190, 221]
input_ramp = [902, 904]
output_ramp = [901, 903, 905]

enter_line = [23, 121, 190]
input_main_ramp = [121, 190]
output_main_ramp = [73, 126, 220]

three_lane = [71, 72, 73, 119, 120, 125, 126, 188, 189, 216, 217, 218, 219, 220]

# 전역 변수 (STVM 함수에서 사용)
sec_time = SEC_TIME
revision_time = 3600 / sec_time

detector_map = {}


# =============================================================================
# 2. 공통 함수
# =============================================================================
def sort_ascending(df):
    df = df.copy()
    df["StartTime"] = (
        df["TimeGroup"].astype(str).str.split("~").str[0].astype(int)
    )
    df["EndTime"] = (
        df["TimeGroup"].astype(str).str.split("~").str[1].astype(int)
    )
    df = df.sort_values(["StartTime", "EndTime", "New_Measurement"]).reset_index(drop=True)
    return df


def complete_time_measurement(df, value_cols, measurements=None):
    df = df.copy()

    full_time = pd.DataFrame({
        "StartTime": np.arange(start_interval, end_interval, sec_time)
    })
    full_time["EndTime"] = full_time["StartTime"] + sec_time
    full_time["TimeGroup"] = (
        full_time["StartTime"].astype(str) + "~" + full_time["EndTime"].astype(str)
    )

    if measurements is None:
        measurements = sorted(df["New_Measurement"].dropna().unique())

    measurement_df = pd.DataFrame({"New_Measurement": measurements})

    full_index = (
        full_time.assign(key=1)
        .merge(measurement_df.assign(key=1), on="key")
        .drop(columns="key")
    )

    df = full_index.merge(
        df,
        on=["TimeGroup", "New_Measurement"],
        how="left"
    )

    for col in value_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df = sort_ascending(df)
    return df


def add_time_group(original_df):
    original_df = original_df.copy()

    bins = np.arange(start_interval, end_interval + 1, sec_time)
    labels = [f"{start}~{start + sec_time}" for start in bins[:-1]]

    original_df["TimeGroup"] = pd.cut(
        original_df["t(Entry)"],
        bins=bins,
        labels=labels,
        right=False
    )

    original_df = original_df[original_df["TimeGroup"].notna()].copy()
    original_df["TimeGroup"] = original_df["TimeGroup"].astype(str)

    return original_df


# =============================================================================
# 3. 지표 계산 함수
# =============================================================================
def speed_mean(original_df):
    copy_df = original_df.copy()
    copy_df = copy_df[~copy_df["New_Measurement"].between(900, 910)]

    measurements = sorted(copy_df["New_Measurement"].dropna().unique())

    speed_mean_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .agg(
            V_mean=("v[km/h]", "mean"),
            V_count=("v[km/h]", "count")
        )
        .reset_index()
    )

    speed_mean_df = complete_time_measurement(
        speed_mean_df,
        value_cols=["V_mean", "V_count"],
        measurements=measurements
    )

    # 다음 검지기 속도
    speed_mean_df["V_next"] = speed_mean_df.groupby("TimeGroup")["V_mean"].shift(-1)

    cols = ["V_mean", "V_next"]
    speed_mean_df[cols] = speed_mean_df[cols].fillna(0)

    speed_mean_df["delta_V"] = np.where(
        speed_mean_df["V_mean"] == 0,
        0,
        (speed_mean_df["V_next"] - speed_mean_df["V_mean"]) / speed_mean_df["V_mean"]
    )

    return speed_mean_df


def density_mean(speed_df):
    density_mean_df = speed_df.copy()

    density_mean_df["K"] = np.where(
        density_mean_df["V_mean"] == 0,
        0,
        density_mean_df["V_count"] * revision_time / density_mean_df["V_mean"]
    )

    density_mean_df["K_next"] = density_mean_df.groupby("TimeGroup")["K"].shift(-1)

    cols = ["K", "K_next"]
    density_mean_df[cols] = density_mean_df[cols].fillna(0)

    density_mean_df["delta_K"] = np.where(
        density_mean_df["K"] == 0,
        0,
        (density_mean_df["K_next"] - density_mean_df["K"]) / density_mean_df["K"]
    )

    density_mean_df = complete_time_measurement(
        density_mean_df,
        value_cols=["V_mean", "V_count", "K", "K_next", "delta_K"]
    )

    return density_mean_df


def heavy_rate(original_df):
    copy_df = original_df.copy()
    measurements = sorted(copy_df["New_Measurement"].dropna().unique())

    heavy_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .agg(
            heavy_count=("Vehicle type", lambda x: x.isin([630, 640, 650]).sum()),
            total_count=("Vehicle type", "count")
        )
        .reset_index()
    )

    heavy_df = sort_ascending(heavy_df)

    heavy_df = complete_time_measurement(
        heavy_df,
        value_cols=["heavy_count", "total_count"],
        measurements=measurements
    )

    heavy_df["rate"] = np.where(
        heavy_df["total_count"] == 0,
        0,
        heavy_df["heavy_count"] / heavy_df["total_count"]
    )

    return heavy_df


def entry_saturation(original_df):
    copy_df = original_df.copy()
    copy_df = copy_df[~copy_df["New_Measurement"].between(900, 910)]

    measurements = sorted(copy_df["New_Measurement"].dropna().unique())

    entry_saturation_df = (
        copy_df.groupby(["TimeGroup", "New_Measurement"])
        .size()
        .reset_index(name="entry_volume")
    )

    entry_saturation_df = complete_time_measurement(
        entry_saturation_df,
        value_cols=["entry_volume"],
        measurements=measurements
    )

    entry_saturation_df = sort_ascending(entry_saturation_df)

    entry_saturation_df["capacity"] = np.where(
        entry_saturation_df["New_Measurement"].isin(three_lane),
        6600,
        4400
    )

    entry_saturation_df["Phi_진입"] = (
        entry_saturation_df["entry_volume"] * revision_time / entry_saturation_df["capacity"]
    )

    return entry_saturation_df


def rfr_rate(original_df):
    copy_df = original_df.copy()
    copy_df = copy_df[copy_df["TimeGroup"].notna()]
    copy_df["TimeGroup"] = copy_df["TimeGroup"].astype(str)

    main_results = []

    for before, after in zip(before_ramp, after_ramp):
        q_before = (
            copy_df[copy_df["New_Measurement"] == before]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="q_before")
        )

        q_after = (
            copy_df[copy_df["New_Measurement"] == after]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="q_after")
        )

        merged = q_before.merge(q_after, on="TimeGroup", how="outer").fillna(0)
        merged["before_ramp"] = before
        merged["after_ramp"] = after
        merged["Qm"] = (merged["q_before"] + merged["q_after"]) / 2
        main_results.append(merged)

    ramp_results = []

    for input_, output_ in zip_longest(input_ramp, output_ramp):
        if output_ is not None:
            q_out = (
                copy_df[copy_df["New_Measurement"] == output_]
                .groupby("TimeGroup")
                .size()
                .reset_index(name="q_out")
            )
            q_out["New_Measurement"] = output_
            ramp_results.append(q_out)

        if input_ is not None:
            q_in = (
                copy_df[copy_df["New_Measurement"] == input_]
                .groupby("TimeGroup")
                .size()
                .reset_index(name="q_in")
            )
            q_in["New_Measurement"] = input_
            ramp_results.append(q_in)

    input_queue = input_main_ramp.copy()
    output_queue = output_main_ramp.copy()
    rfr_list = []

    for i in range(min(len(main_results), len(ramp_results))):
        main_df = main_results[i]
        ramp_df = ramp_results[i]

        rfr_df = pd.merge(main_df, ramp_df, on="TimeGroup", how="outer").fillna(0)

        rfr_df["IR_in"] = 0
        rfr_df["IR_out"] = 0

        if "q_in" in rfr_df.columns:
            rfr_df["IR_in"] = np.where(
                rfr_df["Qm"] == 0,
                0,
                rfr_df["q_in"] / rfr_df["Qm"]
            )
            if input_queue:
                current_input = input_queue.pop(0)
                rfr_df["New_Measurement"] = current_input

        if "q_out" in rfr_df.columns:
            rfr_df["IR_out"] = np.where(
                rfr_df["Qm"] == 0,
                0,
                rfr_df["q_out"] / rfr_df["Qm"]
            )
            if output_queue:
                current_output = output_queue.pop(0)
                rfr_df["New_Measurement"] = current_output

        rfr_df["RFR"] = rfr_df["IR_in"] + rfr_df["IR_out"]
        rfr_list.append(rfr_df)

    if not rfr_list:
        base = copy_df[["TimeGroup"]].drop_duplicates().copy()
        all_measurements = copy_df["New_Measurement"].unique()

        expanded = []
        for m in all_measurements:
            temp = base.copy()
            temp["New_Measurement"] = m
            temp["RFR"] = 0
            expanded.append(temp)

        final_rfr_df = pd.concat(expanded, ignore_index=True)
        final_rfr_df = final_rfr_df.sort_values(["TimeGroup", "New_Measurement"]).reset_index(drop=True)
        final_rfr_df["RFR"] = final_rfr_df["RFR"].fillna(0)

    else:
        final_rfr_df = pd.concat(rfr_list, ignore_index=True)

        target_measurements = input_main_ramp + output_main_ramp
        all_measurements = copy_df["New_Measurement"].unique()

        expanded_df_list = []
        base_rfr_df = final_rfr_df.copy()

        for m in all_measurements:
            if m in target_measurements:
                temp = base_rfr_df[base_rfr_df["New_Measurement"] == m].copy()
            else:
                temp = base_rfr_df[["TimeGroup"]].drop_duplicates().copy()
                temp["New_Measurement"] = m
                temp["RFR"] = 0

            expanded_df_list.append(temp)

        final_rfr_df = pd.concat(expanded_df_list, ignore_index=True)
        final_rfr_df = final_rfr_df.sort_values(["TimeGroup", "New_Measurement"]).reset_index(drop=True)
        final_rfr_df = final_rfr_df[["TimeGroup", "New_Measurement", "RFR"]]
        final_rfr_df["RFR"] = final_rfr_df["RFR"].fillna(0)

    final_rfr_df = sort_ascending(final_rfr_df)

    final_rfr_df = complete_time_measurement(
        final_rfr_df,
        value_cols=["RFR"]
    )

    return final_rfr_df


def output_normality(original_df):
    copy_df = original_df.copy()
    copy_df = copy_df[copy_df["TimeGroup"].notna()]

    normality_list = []

    for enter, exit_ramp, exit_main in zip(enter_line, output_ramp, output_main_ramp):
        entry_df = copy_df[copy_df["New_Measurement"] == enter][
            ["New_Measurement", "VehNo", "t(Entry)"]
        ]

        exit_df = copy_df[copy_df["New_Measurement"] == exit_ramp][
            ["New_Measurement", "VehNo", "t(Entry)"]
        ]

        entry_first = (
            entry_df.groupby("VehNo")["t(Entry)"]
            .min()
            .reset_index()
            .rename(columns={"t(Entry)": "t_entry"})
        )

        exit_first = (
            exit_df.groupby("VehNo")["t(Entry)"]
            .min()
            .reset_index()
            .rename(columns={"t(Entry)": "t_exit"})
        )

        merged = pd.merge(entry_first, exit_first, on="VehNo", how="inner")
        merged["delay_sec"] = merged["t_exit"] - merged["t_entry"]
        merged = merged[merged["delay_sec"] >= 0]

        if len(merged) and np.isfinite(np.nanmedian(merged["delay_sec"])):
            lag_bins = int(round(np.nanmedian(merged["delay_sec"]) / sec_time))
        else:
            lag_bins = 0

        entry_count = (
            copy_df[copy_df["New_Measurement"] == enter]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="Q_in")
        )

        exit_count = (
            copy_df[copy_df["New_Measurement"] == exit_ramp]
            .groupby("TimeGroup")
            .size()
            .reset_index(name="Q_out")
        )

        merged_counts = pd.merge(entry_count, exit_count, on="TimeGroup", how="left")
        merged_counts["Q_out_shift"] = merged_counts["Q_out"].shift(-lag_bins)

        merged_counts["F(outrate)"] = np.where(
            merged_counts["Q_in"] == 0,
            0,
            merged_counts["Q_out_shift"] / merged_counts["Q_in"]
        )

        merged_counts["New_Measurement"] = exit_main
        normality_list.append(merged_counts)

    if not normality_list:
        base = copy_df[["TimeGroup"]].drop_duplicates().copy()
        all_measurements = copy_df["New_Measurement"].unique()

        expanded = []
        for m in all_measurements:
            temp = base.copy()
            temp["New_Measurement"] = m
            temp["F(outrate)"] = 0
            expanded.append(temp)

        final_df = pd.concat(expanded, ignore_index=True)
        final_df = final_df.sort_values(["TimeGroup", "New_Measurement"]).reset_index(drop=True)
        final_df["F(outrate)"] = final_df["F(outrate)"].fillna(0)

    else:
        final_df = pd.concat(normality_list, ignore_index=True)

        all_measurements = copy_df["New_Measurement"].unique()
        expanded_list = []

        for m in all_measurements:
            if m in output_main_ramp:
                temp = final_df[final_df["New_Measurement"] == m].copy()
            else:
                temp = final_df[["TimeGroup"]].drop_duplicates().copy()
                temp["New_Measurement"] = m
                temp["F(outrate)"] = 0

            expanded_list.append(temp)

        final_df = pd.concat(expanded_list, ignore_index=True)
        final_df = final_df.sort_values(["TimeGroup", "New_Measurement"]).reset_index(drop=True)

    final_df = sort_ascending(final_df)

    final_df = complete_time_measurement(
        final_df,
        value_cols=["F(outrate)"]
    )

    return final_df


def calculate_stvm(speed_df, density_df, heavy_df, entry_saturation_df, rfr_df, normality_df):
    merged_df = (
        speed_df[["TimeGroup", "New_Measurement", "delta_V"]]
        .merge(
            density_df[["TimeGroup", "New_Measurement", "delta_K"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            heavy_df[["TimeGroup", "New_Measurement", "rate"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            entry_saturation_df[["TimeGroup", "New_Measurement", "Phi_진입"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            rfr_df[["TimeGroup", "New_Measurement", "RFR"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
        .merge(
            normality_df[["TimeGroup", "New_Measurement", "F(outrate)"]],
            on=["TimeGroup", "New_Measurement"],
            how="outer"
        )
    )

    merged_df.fillna(0, inplace=True)

    merged_df["STVM"] = (
        weights["w1"] * merged_df["delta_V"]
        + weights["w2"] * merged_df["delta_K"]
        + weights["w3"] * merged_df["rate"]
        + weights["w4"] * merged_df["Phi_진입"]
        + weights["w5"] * merged_df["RFR"]
        + weights["w6"] * merged_df["F(outrate)"]
    )

    merged_df.replace([np.inf, -np.inf], 0, inplace=True)
    merged_df = modify_frame(merged_df)

    return merged_df


def modify_frame(df):
    modify_df = df.copy()

    if "StartTime" not in modify_df.columns or "EndTime" not in modify_df.columns:
        modify_df["StartTime"] = modify_df["TimeGroup"].str.split("~").str[0].astype(int)
        modify_df["EndTime"] = modify_df["TimeGroup"].str.split("~").str[1].astype(int)

    modify_df = modify_df[
        (modify_df["StartTime"] >= start_interval)
        & (modify_df["EndTime"] <= end_use_interval)
    ]

    modify_df = modify_df[~modify_df["New_Measurement"].isin([266, 901, 902, 903, 904, 905])]
    modify_df = modify_df.sort_values(["StartTime", "New_Measurement"]).reset_index(drop=True)

    return modify_df


# =============================================================================
# 4. processed parquet 생성
# =============================================================================
def build_processed_file(raw_file_path, processed_file_path):
    use_cols = ["Measurem.", "t(Entry)", "v[km/h]", "Vehicle type", "VehNo"]

    df = pd.read_parquet(raw_file_path, columns=use_cols)
    df = df.apply(pd.to_numeric, errors="coerce")

    original_df = df[df["t(Entry)"] != -1.00].reset_index(drop=True)
    original_df.drop(columns=["b[m/s2]", "tQueue", "Occ", "Pers"], inplace=True, errors="ignore")
    original_df["New_Measurement"] = original_df["Measurem."] % 1000

    original_df = add_time_group(original_df)

    speed_df = speed_mean(original_df)
    density_df = density_mean(speed_df)
    heavy_df = heavy_rate(original_df)
    entry_df = entry_saturation(original_df)
    rfr_df = rfr_rate(original_df)
    normality_df = output_normality(original_df)
    stvm_df = calculate_stvm(speed_df, density_df, heavy_df, entry_df, rfr_df, normality_df)

    processed_df = (
        speed_df[[
            "StartTime", "EndTime", "TimeGroup", "New_Measurement",
            "V_mean", "V_count", "V_next", "delta_V"
        ]]
        .merge(
            density_df[["TimeGroup", "New_Measurement", "K", "K_next", "delta_K"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            heavy_df[["TimeGroup", "New_Measurement", "rate"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            entry_df[["TimeGroup", "New_Measurement", "Phi_진입"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            rfr_df[["TimeGroup", "New_Measurement", "RFR"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            normality_df[["TimeGroup", "New_Measurement", "F(outrate)"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
        .merge(
            stvm_df[["TimeGroup", "New_Measurement", "STVM"]],
            on=["TimeGroup", "New_Measurement"],
            how="left"
        )
    )

    processed_df = modify_frame(processed_df)
    processed_df = processed_df.fillna(0)

    processed_df.to_parquet(processed_file_path, index=False)

    del df, original_df
    del speed_df, density_df, heavy_df, entry_df, rfr_df, normality_df, stvm_df, processed_df
    gc.collect()


def build_processed_files(folder_path, parquet_list, processed_folder, overwrite=False):
    os.makedirs(processed_folder, exist_ok=True)

    for mer_file in parquet_list:
        raw_file_path = os.path.join(folder_path, mer_file)
        processed_name = f"processed_{sec_time}sec_{mer_file}"
        processed_file_path = os.path.join(processed_folder, processed_name)

        if os.path.exists(processed_file_path) and not overwrite:
            continue

        # print(f"processed 생성: {processed_name}")
        build_processed_file(raw_file_path, processed_file_path)


# =============================================================================
# 5. 정규화 (Z-score → 환산점수)
# =============================================================================
# z_to_score: 1. stvm__hwaseong_seoul_V1.0.ipynb 그대로 (보고서 표21)
def z_to_score(z, z_min, z_max):
    if 1.645 <= z <= z_max:
        return 50 + ((95 + 5 * ((z - 1.645) / (z_max - 1.645))) * 0.5)
    elif 1.282 <= z < 1.645:
        return 50 + ((90 + 5 * ((z - 1.282) / (1.645 - 1.282))) * 0.5)
    elif 1.038 <= z < 1.282:
        return 50 + ((85 + 5 * ((z - 1.038) / (1.282 - 1.038))) * 0.5)
    elif 0.842 <= z < 1.038:
        return 50 + ((80 + 5 * ((z - 0.842) / (1.038 - 0.842))) * 0.5)
    elif 0.676 <= z < 0.842:
        return 50 + ((75 + 5 * ((z - 0.676) / (0.842 - 0.676))) * 0.5)
    elif 0.526 <= z < 0.676:
        return 50 + ((70 + 5 * ((z - 0.526) / (0.676 - 0.526))) * 0.5)
    elif 0.387 <= z < 0.526:
        return 50 + ((65 + 5 * ((z - 0.387) / (0.526 - 0.387))) * 0.5)
    elif 0.255 <= z < 0.387:
        return 50 + ((60 + 5 * ((z - 0.255) / (0.387 - 0.255))) * 0.5)
    elif -0.255 <= z < 0.255:
        return 50 + ((40 + 5 * ((z + 0.255) / (0.255 + 0.255))) * 0.5)
    elif -0.387 <= z < -0.255:
        return 50 + ((35 + 5 * ((z + 0.387) / (-0.255 + 0.387))) * 0.5)
    elif -0.526 <= z < -0.387:
        return 50 + ((30 + 5 * ((z + 0.526) / (-0.387 + 0.526))) * 0.5)
    elif -0.676 <= z < -0.526:
        return 50 + ((25 + 5 * ((z + 0.676) / (-0.676 + 0.842))) * 0.5)
    elif -0.842 <= z < -0.676:
        return 50 + ((20 + 5 * ((z + 0.842) / (-0.676 + 0.842))) * 0.5)
    elif -1.038 <= z < -0.842:
        return 50 + ((15 + 5 * ((z + 1.038) / (-0.842 + 1.038))) * 0.5)
    elif -1.282 <= z < -1.038:
        return 50 + ((10 + 5 * ((z + 1.282) / (-1.038 + 1.282))) * 0.5)
    elif -1.645 <= z < -1.282:
        return 50 + ((5 + 5 * ((z + 1.645) / (-1.282 + 1.645))) * 0.5)
    elif z_min <= z < -1.645:
        return 50 + ((0 + 5 * ((z + z_min) / (-1.645 + z_min))) * 0.5)
    else:
        return np.nan
# ---- end verbatim ----


def calc_z_timeseries(stvm_series):
    """시나리오별 정규화: 시나리오 전체(전 위치 x 전 시간) STVM의 평균·STDEV.P로 Z,
    그 시나리오의 Z최소·Z최대로 z_to_score(표21) 환산."""
    s = stvm_series.astype(float)
    mu = s.mean()
    sd = s.std(ddof=0)
    z = (s - mu) / sd
    z_min, z_max = z.min(), z.max()
    score = z.apply(lambda v: z_to_score(v, z_min, z_max))
    info = {"STVM_평균": mu, "STVM_표준편차": sd, "Z최소": z_min, "Z최대": z_max, "값개수": len(s)}
    return z, score, info


# =============================================================================
# 6. 실행부
# =============================================================================
def run_no(file_name):
    # 파일명 끝 실행 번호(_1 ~ _1400)로 정렬해야 _1000이 _100 뒤에 오지 않음
    return int(re.search(r"_(\d+)\.parquet$", file_name).group(1))


def build_one(args):
    raw_file_path, processed_file_path = args
    if not os.path.exists(processed_file_path):
        part = processed_file_path + ".partial.parquet"
        build_processed_file(raw_file_path, part)
        os.replace(part, processed_file_path)
    return processed_file_path


def scenario_stvm(sid, processed_paths):
    stvm_list = [
        pd.read_parquet(p)[["StartTime", "EndTime", "TimeGroup", "New_Measurement", "STVM"]]
        for p in processed_paths
    ]
    merged_stvm = pd.concat(stvm_list, ignore_index=True)

    # 시드 평균 (기존 코드와 같은 방식)
    stvm_avg = (
        merged_stvm
        .groupby(["TimeGroup", "New_Measurement"], sort=False)
        .mean(numeric_only=True)
        .reset_index()
    )
    stvm_avg = stvm_avg.sort_values(["New_Measurement", "StartTime"]).reset_index(drop=True)

    z, score, info = calc_z_timeseries(stvm_avg["STVM"])

    out_df = pd.DataFrame({
        "scenario_id": sid,
        "time_sec": stvm_avg["StartTime"].astype(int),
        "detector_location_id": "LOC" + stvm_avg["New_Measurement"].astype(int).astype(str).str.zfill(3),
        "STVM": stvm_avg["STVM"],
        "Z_score": z,
        "환산점수": score,
        "aggregation_sec": sec_time,
        "run_type": "base",
        "notes": f"{num_mer}회 평균",
    })
    return out_df, info


if __name__ == "__main__":
    t0 = time.time()

    for set_name in RUN_SETS:
        for sub, csv_name, suffix in SECTIONS:
            if RUN_SECTIONS and sub not in RUN_SECTIONS:
                continue
            scn_name = sub

            folder_path = os.path.join(RAW_ROOT, SET_DIRS[set_name], sub)
            senario_path = os.path.join(SCN_ROOT, SET_DIRS[set_name], csv_name)
            for p in (folder_path, senario_path):
                if not os.path.exists(p):
                    raise FileNotFoundError(f"경로 없음 (RAW_ROOT·SCN_ROOT 설정 확인): {p}")
            senario_df = pd.read_csv(senario_path, encoding="cp949")
            senario_df = senario_df.sort_values("scenario_id").reset_index(drop=True)

            parquet_list = sorted(
                [f for f in os.listdir(folder_path)
                 if f.endswith(".parquet") and not f.startswith("processed_")],
                key=run_no
            )
            assert [run_no(f) for f in parquet_list] == list(range(1, len(parquet_list) + 1)), \
                f"실행 번호가 빠진 파일이 있음: {folder_path}"
            assert len(parquet_list) == len(senario_df) * num_mer, \
                f"파일 {len(parquet_list)}개 ≠ 시나리오 {len(senario_df)}개 x {num_mer}: {folder_path}"

            result_folder = os.path.join(folder_path, RESULT_FOLDER_NAME)          # parquet 폴더\STVM 시계열
            processed_folder = os.path.join(result_folder, "_processed")          # 중간 파일 (계산 후 삭제)
            os.makedirs(processed_folder, exist_ok=True)
            section_stats = []

            for idx, start in enumerate(range(0, len(parquet_list), num_mer)):
                sid = int(senario_df.iloc[idx]["scenario_id"])
                if RUN_SCENARIOS and sid not in RUN_SCENARIOS:
                    continue

                out_path = os.path.join(result_folder, f"Scenario_{sid:03d}_{num_mer}회평균_STVM_{suffix}.csv")
                if os.path.exists(out_path):
                    print(f"[건너뜀] {set_name} {scn_name} 시나리오 {sid} (이미 있음)")
                    continue

                batch_files = parquet_list[start:start + num_mer]
                jobs = [
                    (os.path.join(folder_path, f),
                     os.path.join(processed_folder, f"processed_{sec_time}sec_{f}"))
                    for f in batch_files
                ]

                # 원시 파일별 processed 생성 (기존 build_processed_file 그대로)
                if N_WORKERS > 1:
                    processed_paths = Parallel(n_jobs=min(N_WORKERS, len(jobs)), backend="loky")(
                        delayed(build_one)(j) for j in jobs)
                else:
                    processed_paths = [build_one(j) for j in jobs]

                out_df, info = scenario_stvm(sid, processed_paths)
                out_df.to_csv(out_path + ".partial", index=False, encoding="utf-8-sig")
                os.replace(out_path + ".partial", out_path)
                section_stats.append({"세트": set_name, "구간": scn_name, "scenario_id": sid, **info})

                if DELETE_PROCESSED:
                    for p in processed_paths:
                        os.remove(p)

                print(f"[{datetime.now():%H:%M:%S}] {set_name} {scn_name} 시나리오 {sid:03d} 완료 "
                      f"(누적 {(time.time() - t0) / 60:.1f}분)", flush=True)
                gc.collect()

            # 비어 있는 중간 파일 폴더 정리
            if DELETE_PROCESSED and not os.listdir(processed_folder):
                os.rmdir(processed_folder)
            """
            # 정규화 기준값 표: 구간 폴더마다 1개 (나눠서 실행해도 기존 내용에 이어서 저장)
            if section_stats:
                stat_path = os.path.join(result_folder, f"정규화_기준값_{suffix}.csv")
                new_stats = pd.DataFrame(section_stats)
                if os.path.exists(stat_path):
                    new_stats = pd.concat([pd.read_csv(stat_path, encoding="utf-8-sig"), new_stats], ignore_index=True)
                new_stats = (new_stats.drop_duplicates("scenario_id", keep="last")
                             .sort_values("scenario_id").reset_index(drop=True))
                new_stats.to_csv(stat_path, index=False, encoding="utf-8-sig")
                print("기준값 표 저장:", stat_path)
            """

[12:59:52] 추정용 진입구간 시나리오 001 완료 (누적 0.6분)
기준값 표 저장: E:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\시나리오(140-추정용,검증용)_261022\1. 추정용 시나리오\진입구간\STVM 시계열\정규화_기준값_진입부.csv
